In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [2]:
%%capture
# Install openai_harmony (Harmony protocol tools) if missing
import importlib.util, sys, subprocess

if importlib.util.find_spec("openai_harmony") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai-harmony"])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai_harmony"])

In [3]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = 4096,
    #load_in_4bit = True, # False for LoRA 16bit
    #offload_embedding = True, # Reduces VRAM by 1GB
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
messages = [
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : "2"},
    {"role": "user",  "content": "What's the temperature in San Francisco now? How about tomorrow? Today's date is 2024-09-30."},
    {"role": "assistant",  "content": "User asks: 'What is the weather in San Francisco?' We need to use get_current_temperature tool.", "thinking" : ""},
    {"role": "assistant", "content": "", "tool_calls": [{"name": "get_current_temperature", "arguments": '{"location": "San Francisco, California, United States", "unit": "celsius"}'}]},
    {"role": "tool", "name": "get_current_temperature", "content": '{"temperature": 19.9, "location": "San Francisco, California, United States", "unit": "celsius"}'},
]

In [4]:
from unsloth_zoo import encode_conversations_with_harmony



('<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-12-29\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What is 1+1?<|end|><|start|>assistant<|channel|>final<|message|>2<|end|><|start|>user<|message|>What\'s the temperature in San Francisco now? How about tomorrow? Today\'s date is 2024-09-30.<|end|><|start|>assistant<|channel|>analysis<|message|>User asks: \'What is the weather in San Francisco?\' We need to use get_current_temperature tool.<|end|><|start|>assistant to=functions.get_current_temperature<|channel|>commentary json<|message|>{"location": "San Francisco, California, United States", "unit": "celsius"}<|call|><|start|>functions.get_current_temperature to=assistant<|channel|>commentary<|message|>{"temperature": 19.9, "location": "San Francisco, California, United States", "unit": "celsius"}<|end|><

In [5]:
messages = [
    {"role" : "user", "content" : "What is 2**100 mod 100?"},
]

In [6]:
A = encode_conversations_with_harmony(
    messages,
    reasoning_effort = "medium",
    add_generation_prompt = True,
    tool_calls = None,
    developer_instructions = None,
    model_identity = "You Do math problems accurately . Provide final answer in this box  ANS[] at the end ",
)

In [ ]:
from transformers import TextStreamer
import torch

# Optional: enables faster inference paths in Unsloth where supported
try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
except Exception:
    pass

device = getattr(model, "device", torch.device("cpu"))

# Build model inputs from your `messages`
# Prefer the tokenizer's chat template; fall back to Harmony-encoded `A` if needed.
inputs = None
chat_template_error = None
try:
    templated = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if isinstance(templated, dict):
        inputs = {k: v.to(device) for k, v in templated.items()}
    else:
        inputs = {"input_ids": templated.to(device)}
except Exception as e:
    chat_template_error = e

if inputs is None:
    if isinstance(A, (tuple, list)) and len(A) >= 1:
        input_ids = A[0]
        attention_mask = A[1] if len(A) >= 2 else None

        if hasattr(input_ids, "to"):
            input_ids = input_ids.to(device)
        inputs = {"input_ids": input_ids}
        if attention_mask is not None and hasattr(attention_mask, "to"):
            inputs["attention_mask"] = attention_mask.to(device)
    else:
        raise RuntimeError(
            "Could not build inputs from tokenizer chat template or Harmony encoding. "
            f"Chat template error: {chat_template_error!r}"
        )

# Stream tokens as they are generated (nice UX), but ALSO decode final output (reliable)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# If the model stops too early, it's usually because it sampled EOS quickly.
# These settings make it much less likely to "stop mid-thought".
gen_kwargs = dict(
    max_new_tokens=512,
    min_new_tokens=64,
    do_sample=False,  # set True if you want randomness
    streamer=streamer,
    return_dict_in_generate=True,
    output_scores=False,
    pad_token_id=(tokenizer.eos_token_id if getattr(tokenizer, "pad_token_id", None) is None else tokenizer.pad_token_id),
)

with torch.no_grad():
    out = model.generate(**inputs, **gen_kwargs)

# Decode only the newly generated tokens (after the prompt) so you always see the full completion.
prompt_len = inputs["input_ids"].shape[-1]
completion_ids = out.sequences[0, prompt_len:]
print("\n\n--- decoded completion (reliable) ---\n")
print(tokenizer.decode(completion_ids, skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


analysisWe need to compute 2^100 mod 100. We can use Euler's theorem: φ(100)=40. Since 2 and 100 not coprime? 2 and 100 share gcd 2. So Euler's theorem doesn't apply directly. But we can compute using Chinese remainder theorem: mod 100 = mod 4 and mod 25.

Compute 2^100 mod 4: 2^1 mod 4 = 2, 2^2 = 0 mod 4? Actually 4 divides 2^2=4, so 2^2 mod 4 = 0. For any exponent >=2, 2^n mod 4 = 0. So 2^100 mod 4 = 0.

Compute 2^100 mod 25: Since 2 and 25 are coprime. φ(25)=20. So 2^20 ≡1 mod 25. So 2^100 = (2^20)^5 = 1^5 = 1 mod 25. So 2^100 ≡1 mod 25.

Now we need x such that x ≡ 0 mod 4 and x ≡ 1 mod
